In [49]:
import os
from fastapi import FastAPI
import mlflow
import socket
from contextlib import asynccontextmanager
from pathlib import Path
from pydantic import BaseModel
import numpy as np
import uvicorn


In [50]:
MODEL_PATH = Path(os.environ.get("MODEL_PATH", "./model"))
POD_NAME = socket.gethostname()
state = {"model": None, "version": None}



In [51]:
MODEL_PATH

PosixPath('model')

In [52]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    state['model'] = mlflow.pyfunc.load_model(MODEL_PATH)
    version_file = MODEL_PATH / "VERSION"
    if version_file.exists():
        with open(version_file, "r") as f:
            state['version'] = f.read().strip()
    else:
        state['version'] = "unknown"
    yield

app = FastAPI(title = "iris_classifier",lifespan=lifespan)


In [53]:
async with lifespan(app):
    print("model:  ", type(state["model"]))
    print("version:", state["version"])
    print("pod:    ", POD_NAME)

    # quick sanity check that the loaded model actually predicts
    print("predict:", state["model"].predict(np.array([[5.1, 3.5, 1.4, 0.2]])))

model:   <class 'mlflow.pyfunc.PyFuncModel'>
version: unknown
pod:     Alluris-MacBook-Air.local
predict: [0]


In [54]:
class InputData(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

In [62]:
@app.get("/")
def info():
    return {"model_version": state["version"], "pod_name": POD_NAME}

@app.get("/ready")
def ready():
    if state["model"] is not None:
        return {"status": "ready"}
    return {"status": "not ready"}

@app.post("/predict")
def predict(input_data: InputData):
    input_array = np.array([[input_data.sepal_length, input_data.sepal_width, input_data.petal_length, input_data.petal_width]])
    prediction = state["model"].predict(input_array)
    return {"prediction": prediction}



In [63]:
predict(InputData(sepal_length=5.1, sepal_width=3.5, petal_length=1.4, petal_width=0.2))


{'prediction': array([0])}

In [56]:
uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=True)

INFO:     Will watch for changes in these directories: ['/Users/allurijairam/Desktop/Code_FRF/Kubernetes /app']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [61770] using StatReload
ERROR:    Error loading ASGI app. Could not import module "main".
INFO:     Stopping reloader process [61770]
